# Holdout Forensic Audit

**Source script:** `holdout_forensic_audit.py`

Notebook mirror for submission traceability.

In [ ]:
from __future__ import annotations

import argparse
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

from run_eda_v3 import (
    BINARY_DISEASE_CLASSES,
    SEED,
    add_moon_cyclic_features,
    build_feature_blocks,
    classify_columns,
    compute_balanced_sample_weight,
    derive_temporal_columns,
    infer_date_column,
    infer_outcome_column,
    load_dataset,
    make_hist_gb_binary_pipeline,
)

ROOT = Path(__file__).resolve().parent
DEFAULT_INPUT = ROOT / "outputs" / "imputed_dataset_enriched.csv"
DEFAULT_OUTDIR = ROOT / "outputs" / "eda_v3" / "final_validation" / "holdout_audit"


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Holdout forensic audit.")
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT, help="Input enriched dataset")
    parser.add_argument("--outdir", type=Path, default=DEFAULT_OUTDIR, help="Audit output directory")
    parser.add_argument("--corr-threshold", type=float, default=0.85, help="Correlation threshold")
    parser.add_argument("--seed", type=int, default=SEED, help="Random seed")
    parser.add_argument("--bootstrap-n", type=int, default=2000, help="Bootstrap resamples")
    return parser.parse_args()


def correlation_filter_like_modeling(df: pd.DataFrame, weather_numeric_cols: Sequence[str], threshold: float) -> List[str]:
    if not weather_numeric_cols:
        return []
    X = df[list(weather_numeric_cols)].copy()
    X = X.apply(pd.to_numeric, errors="coerce")
    X = X.fillna(X.median(numeric_only=True))
    corr = X.corr(numeric_only=True).abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if (upper[col] > threshold).any()]
    return [c for c in weather_numeric_cols if c not in to_drop]


def build_aligned_feature_blocks(df: pd.DataFrame, corr_threshold: float) -> Tuple[pd.DataFrame, str, Dict[str, Dict[str, List[str]]], Dict[str, List[str]]]:
    drop_weather_reason = [c for c in df.columns if "weather_missing_reason" in c.lower()]
    if drop_weather_reason:
        df = df.drop(columns=drop_weather_reason)

    outcome_col = infer_outcome_column(df)
    date_col = infer_date_column(df)
    df, temporal_cols = derive_temporal_columns(df, date_col)
    df, _ = add_moon_cyclic_features(df)
    col_groups = classify_columns(df, outcome_col, temporal_cols)

    kept_weather_numeric = correlation_filter_like_modeling(
        df=df,
        weather_numeric_cols=col_groups["weather_model_numeric"],
        threshold=corr_threshold,
    )

    feature_blocks = build_feature_blocks(col_groups)
    env_numeric_original = set(col_groups["weather_model_numeric"])
    feature_blocks["environmental_only"]["numeric"] = sorted(kept_weather_numeric)
    feature_blocks["combined"]["numeric"] = sorted(
        [c for c in feature_blocks["combined"]["numeric"] if c not in env_numeric_original]
        + list(kept_weather_numeric)
    )

    cv_env = {
        "numeric": sorted(kept_weather_numeric),
        "categorical": sorted(col_groups["weather_model_categorical"]),
    }
    return df, outcome_col, feature_blocks, cv_env


def save_split_counts(y_train: pd.Series, y_test: pd.Series, out_file: Path) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for split_name, y_split in [("train", y_train), ("test", y_test)]:
        vc = y_split.value_counts(dropna=False)
        total = len(y_split)
        for cls, n in vc.items():
            rows.append(
                {
                    "split": split_name,
                    "class_label": str(cls),
                    "n": int(n),
                    "pct_within_split": float(100.0 * n / total),
                }
            )
    out = pd.DataFrame(rows).sort_values(["split", "class_label"]).reset_index(drop=True)
    out.to_csv(out_file, index=False)
    return out


def save_feature_lists(
    cv_env: Dict[str, List[str]],
    holdout_env: Dict[str, List[str]],
    outdir: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    cv_rows = [{"feature_name": c, "feature_type": "numeric"} for c in cv_env["numeric"]] + [
        {"feature_name": c, "feature_type": "categorical"} for c in cv_env["categorical"]
    ]
    hold_rows = [{"feature_name": c, "feature_type": "numeric"} for c in holdout_env["numeric"]] + [
        {"feature_name": c, "feature_type": "categorical"} for c in holdout_env["categorical"]
    ]
    cv_df = pd.DataFrame(cv_rows).sort_values(["feature_type", "feature_name"]).reset_index(drop=True)
    hold_df = pd.DataFrame(hold_rows).sort_values(["feature_type", "feature_name"]).reset_index(drop=True)
    cv_df.to_csv(outdir / "cv_env_feature_list.csv", index=False)
    hold_df.to_csv(outdir / "holdout_env_feature_list.csv", index=False)

    cv_set = set(cv_df["feature_name"].tolist())
    hold_set = set(hold_df["feature_name"].tolist())
    all_feats = sorted(cv_set | hold_set)
    diff_rows = []
    for f in all_feats:
        in_cv = f in cv_set
        in_hold = f in hold_set
        if in_cv != in_hold:
            diff_rows.append({"feature_name": f, "in_cv": in_cv, "in_holdout": in_hold})
    diff_df = pd.DataFrame(diff_rows)
    if diff_df.empty:
        diff_df = pd.DataFrame(columns=["feature_name", "in_cv", "in_holdout"])
    diff_df.to_csv(outdir / "env_feature_diff.csv", index=False)
    return cv_df, hold_df, diff_df


def save_leakage_scan(feature_names: Sequence[str], out_file: Path) -> pd.DataFrame:
    bad_tokens = ["group", "diagnos", "outcome", "target"]
    hits = [f for f in feature_names if any(tok in f.lower() for tok in bad_tokens)]
    hit_df = pd.DataFrame({"feature_name": hits})
    hit_df.to_csv(out_file, index=False)
    return hit_df


def bootstrap_auc(y_true: np.ndarray, y_score: np.ndarray, n_boot: int, seed: int) -> Tuple[float, float, float, float]:
    auc = float(roc_auc_score(y_true, y_score))
    rng = np.random.default_rng(seed)
    n = len(y_true)
    vals: List[float] = []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        ys = y_true[idx]
        ps = y_score[idx]
        if len(np.unique(ys)) < 2:
            continue
        vals.append(float(roc_auc_score(ys, ps)))
    if not vals:
        return auc, np.nan, np.nan, np.nan
    arr = np.asarray(vals, dtype=float)
    lo = float(np.percentile(arr, 2.5))
    hi = float(np.percentile(arr, 97.5))
    return auc, lo, hi, float(hi - lo)


def maybe_load_saved_predictions(pred_path: Path) -> pd.DataFrame:
    if pred_path.exists():
        df = pd.read_csv(pred_path)
        req = {"disease", "y_true", "y_proba"}
        if req.issubset(set(df.columns)):
            return df.copy()
    return pd.DataFrame()


def fit_env_holdout_predictions(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    env_num: Sequence[str],
    env_cat: Sequence[str],
    seed: int,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for disease in BINARY_DISEASE_CLASSES:
        y_tr_bin = (y_train == disease).astype(int)
        y_te_bin = (y_test == disease).astype(int)
        if y_tr_bin.nunique() < 2 or y_te_bin.nunique() < 2:
            continue

        pipe = make_hist_gb_binary_pipeline(env_num, env_cat, random_state=seed)
        sw = compute_balanced_sample_weight(y_tr_bin)
        pipe.fit(X_train, y_tr_bin, model__sample_weight=sw)
        proba = pipe.predict_proba(X_test)[:, 1]
        for yt, yp in zip(y_te_bin.to_numpy(), proba):
            rows.append({"disease": disease, "y_true": int(yt), "y_proba": float(yp)})
    return pd.DataFrame(rows)


def main() -> None:
    args = parse_args()
    np.random.seed(args.seed)
    args.outdir.mkdir(parents=True, exist_ok=True)

    df_raw = load_dataset(args.input)
    df, outcome_col, feature_blocks, cv_env = build_aligned_feature_blocks(df_raw, args.corr_threshold)
    env_num = list(feature_blocks["environmental_only"]["numeric"])
    env_cat = list(feature_blocks["environmental_only"]["categorical"])
    env_features = env_num + env_cat


    y = df[outcome_col].astype(str)
    X_env = df[env_features].copy()
    X_train, X_test, y_train, y_test = train_test_split(
        X_env,
        y,
        test_size=0.2,
        random_state=args.seed,
        stratify=y,
    )
    split_counts = save_split_counts(y_train, y_test, args.outdir / "holdout_split_counts.csv")


    holdout_env = {"numeric": sorted(env_num), "categorical": sorted(env_cat)}
    _, _, diff_df = save_feature_lists(cv_env, holdout_env, args.outdir)


    leakage_hits = save_leakage_scan(env_features, args.outdir / "leakage_scan_hits.csv")


    pred_file = args.outdir / "holdout_env_predictions.csv"
    pred_df = maybe_load_saved_predictions(pred_file)
    used_saved_preds = not pred_df.empty
    if pred_df.empty:
        pred_df = fit_env_holdout_predictions(
            X_train=X_train,
            y_train=y_train,
            X_test=X_test,
            y_test=y_test,
            env_num=env_num,
            env_cat=env_cat,
            seed=args.seed,
        )
        pred_df.to_csv(pred_file, index=False)

    auc_rows: List[Dict[str, object]] = []
    variance_flags: List[bool] = []
    unstable_flags: List[bool] = []

    for disease in BINARY_DISEASE_CLASSES:
        sub = pred_df[pred_df["disease"] == disease].copy()
        if sub.empty:
            continue
        y_true = sub["y_true"].to_numpy(dtype=int)
        y_proba = sub["y_proba"].to_numpy(dtype=float)
        n_test_total = int(len(sub))
        n_pos = int(y_true.sum())
        n_neg = int(n_test_total - n_pos)
        high_var_risk = bool(n_pos < 10)
        variance_flags.append(high_var_risk)

        auc, ci_low, ci_high, ci_width = bootstrap_auc(y_true, y_proba, args.bootstrap_n, seed=args.seed)
        unstable = bool(pd.notna(ci_width) and ci_width > 0.15)
        unstable_flags.append(unstable)
        auc_rows.append(
            {
                "disease": disease,
                "n_test_total": n_test_total,
                "n_test_positive": n_pos,
                "n_test_negative": n_neg,
                "high_variance_risk": high_var_risk,
                "auc": auc,
                "ci_low": ci_low,
                "ci_high": ci_high,
                "ci_width": ci_width,
                "unstable": unstable,
            }
        )

    auc_df = pd.DataFrame(auc_rows).sort_values("disease").reset_index(drop=True)
    auc_df.to_csv(args.outdir / "holdout_auc_bootstrap_ci.csv", index=False)


    print("=== HOLDOUT FORENSIC AUDIT ===")
    print(f"total N: {len(df)}")
    print(f"total test N: {len(y_test)}")
    print("per-class test counts:")
    test_counts = split_counts[split_counts["split"] == "test"][["class_label", "n", "pct_within_split"]]
    print(test_counts.to_string(index=False))

    any_small = any(variance_flags)
    print(f"any class with n_test_positive < 10: {any_small}")

    has_diff = not diff_df.empty
    print(f"feature diff detected (CV vs holdout env): {has_diff}")
    if has_diff:
        print(diff_df.to_string(index=False))

    if leakage_hits.empty:
        print("NO LEAKAGE DETECTED")
    else:
        print("LEAKAGE HITS DETECTED:")
        print(leakage_hits.to_string(index=False))

    print(f"used_saved_predictions: {used_saved_preds}")
    print("bootstrap AUC table:")
    print(auc_df.to_string(index=False))
    for _, r in auc_df.iterrows():
        print(f"{r['disease']}: {'Unstable' if bool(r['unstable']) else 'Stable'}")


if __name__ == "__main__":
    main()
